# Notebook 01: Wordle Game Environment & Mechanics
Welcome to the introductory notebook of the **Wordle AI** research project!

### Objectives:
1. Load and inspect the official Wordle solution & allowed guess dictionaries.
2. Demonstrate exact Wordle feedback logic (Gray, Yellow, Green) and repeated-letter edge cases.
3. Walk through interactive game playback with visual board rendering.
4. Demonstrate symbolic constraint candidate filtering.

In [ ]:
import sys
import os

# Dynamic path resolution for local and Kaggle environments
possible_roots = [
    ".",
    "..",
    "/kaggle/working",
    "/kaggle/working/wordle-ai",
    "/kaggle/input/wordle-ai",
]
for root in possible_roots:
    if os.path.exists(os.path.join(root, "src")):
        abs_root = os.path.abspath(root)
        if abs_root not in sys.path:
            sys.path.insert(0, abs_root)
        break

if os.path.exists("/kaggle/input"):
    print("Running on Kaggle GPU/CPU Environment!")
else:
    print("Running on Local Environment!")

from src.wordle.dictionary import WordleDictionary
from src.wordle.game import WordleGame, GameStatus
from src.wordle.feedback import get_feedback, feedback_to_emoji
from src.wordle.constraints import SymbolicConstraints

dict_mgr = WordleDictionary()
print(f"Loaded {len(dict_mgr.secret_words)} secret target words and {len(dict_mgr.all_valid_guesses)} total allowed guess words.")

## 1. Repeated Letter Feedback Verification
Let's inspect the canonical repeated-letter case:
`SECRET = APPLE`, `GUESS = ALLEY`.
- Position 0: `A` == `A` -> 🟩 GREEN
- Position 1: `L` matches `L` in secret -> 🟨 YELLOW
- Position 2: Second `L` in guess has no remaining match -> ⬛ GRAY
- Position 3: `E` matches `E` in secret -> 🟨 YELLOW
- Position 4: `Y` absent -> ⬛ GRAY

In [ ]:
fb = get_feedback("apple", "alley")
print("Feedback digits:", fb)
print("Feedback visual:", feedback_to_emoji(fb))
assert fb == (2, 1, 0, 1, 0)

## 2. Interactive Game Playback & Candidate Filtering

In [ ]:
game = WordleGame(secret_word="crane", dictionary=dict_mgr)
constraints = SymbolicConstraints()

guesses = ["slate", "cater", "crane"]
for g in guesses:
    fb = game.make_guess(g)
    constraints.add_feedback(g, fb)
    rem_cands = constraints.filter_candidates(dict_mgr.secret_words)
    print(f"Guess: {g.upper()} | Feedback: {feedback_to_emoji(fb)} | Candidates Remaining: {len(rem_cands)}")

print("
Final Game Board State:")
print(game.display_board())